# Stage 4 · probes on Llama-3.1-8B-Instruct (multi-turn conversations)

Pipeline: **(1)** generate answers → **(2)** annotate them with the web-search judge → **(3)** train probes on the model's activations →
**(4)** run held-out answers through the probes and compare what lights up with what the judge tagged. This notebook does (3) and (4).

**Data:** our annotated multi-turn conversations from `shreshtha2002/longfact-multiturn-<model>-annotated`.
There are only 60 answers, so **2-fold cross-validation** is used: train on one half, score the other, then swap.
Every answer is scored by a probe that never saw it, split by conversation.
The paper's released English annotations of Llama-3.1-8B answers are added to training (`N_PAPER_TRAIN`); our multi-turn data is the test set.

**Detectors compared**, every one at every layer (0–31):
- **`lin`:** the paper's linear probe on the residual stream, trained with the paper's loss (token BCE annealed into span-max BCE).
- **`mlp`:** a non-linear (MLP) probe on the residual stream, same loss ([Liang & Wang, 2025](https://arxiv.org/abs/2512.20949)).
- **`attn`:** a linear probe on the attention block's output.
- **`head`:** a linear probe on **each attention head's output** (32 heads × 32 layers), giving a map of where the signal is.
- **`LR` / `MM`:** logistic regression / difference of means on entity tokens, at every layer (ARENA 1.3.1; [Marks & Tegmark](https://arxiv.org/abs/2310.06824)).
  LR's L2 strength is picked per layer by AUC on held-out *answers* inside the training fold (tokens of one answer are near-duplicates); MM has no free parameter.
- **`entropy`:** the model's own next-token entropy, which needs no training (baseline; [Phillips et al., 2026](https://arxiv.org/abs/2603.21172)).
- **`paper`:** the paper's released Llama-3.1-8B probe, used as is (Llama only).

**Metrics:** the paper repo's `compute_clf_metrics` at its three levels: all answer tokens, entity tokens, and `span_max` (the paper's headline number).
Each level reports AUC, recall at 10% FPR, accuracy, precision, recall, F1, the best threshold and its accuracy.

**Reused:** the paper repo (losses, span→token search, metrics, label map, token-labelling rules, released training settings) and
ARENA 1.3.1 (`get_pca_components`, `MMProbe`, `LRProbe`, hooks).

**Saving:** a checkpoint every `CKPT_EVERY` texts, plus each finished fold, to `/kaggle/working` and `shreshtha2002/hallucination-probes-multiturn/multiturn-llama`. Run all again to resume.

**Kaggle:** GPU **T4 x2**, Internet **on**, secret **`HF_TOKEN`** (shreshtha2002; it must have accepted the licence for meta-llama/Llama-3.1-8B-Instruct).

In [ ]:
import os, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft", "jaxtyping", "termcolor", "circuitsvis"], check=True)
if not os.path.isdir("repo"):     # the paper's code
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/obalcells/hallucination_probes", "repo"], check=True)
print("ready")

In [ ]:
# ---------------- settings ----------------
SETTING = "multiturn"
MODEL = "meta-llama/Llama-3.1-8B-Instruct"                                     # the HF model we read activations from
DATA_REPO = "shreshtha2002/longfact-multiturn-llama3.1-8b-instruct-q8-0-annotated"   # our judge-annotated answers ("llama3.1-8b-instruct-q8-0" = the Ollama model that wrote them)
PAPER_PROBE = "llama3_1_8b_linear"                         # the paper's released probe for this model (obalcells/hallucination-probes); none exists for Aya
OUT_REPO, RUN = "shreshtha2002/hallucination-probes-multiturn", "multiturn-llama"      # where results go: OUT_REPO/RUN/
N_PAPER_TRAIN = 150                       # paper train rows per source (LongFact, LongFact++) added to training; 0 = ours only
FOLDS = 2                                 # cross-validation folds over our answers (by conversation)
EPOCHS = 2                              # passes over the training texts (the paper uses 1 epoch on far more data)
MAX_LEN = 2048                           # tokens per text (cut from the left; the latest answer is kept)
POS_WEIGHT = NEG_WEIGHT = 10.0            # repo train_config: entity tokens weigh 10, other answer tokens 1
LR, BATCH = 5e-4, 8                       # released llama3_1_8b_linear training_config (probe_head_lr, batch)
MLP_HIDDEN = 64
CACHE_PAPER = 50                          # paper texts whose entity tokens also train LR / MM (all of ours are used)
CKPT_EVERY = 25                           # texts between checkpoints
ATTN_EXAMPLES, ATTN_WINDOW = 8, 1024      # head / attention analysis: answers and tokens per window
SEED = 42
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # less fragmentation (set before torch is imported)
OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "out"
os.makedirs(f"{OUT}/figs", exist_ok=True)

In [ ]:
# ---------------- reused code: the paper's repo + ARENA 1.3.1 ----------------
import gc, html as htmllib, json, math, random, re, time
import numpy as np, pandas as pd, torch, torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from IPython.display import HTML, display
sys.path.insert(0, "repo")
from probe.loss import compute_probe_bce_loss, compute_probe_max_aggregation_loss   # paper's token BCE + span-max losses
from probe.dataset_converters import _MAP_LABEL_TO_SCALAR                          # NS / II -> 1, S -> 0
from utils.tokenization import find_string_in_tokens                                # paper's span -> token search
from utils.metrics import compute_clf_metrics                                       # paper's metrics
t = torch

# ---- ARENA 1.3.1 solutions, verbatim ----
def get_pca_components(activations, k=2):
    X = activations - activations.mean(dim=0)
    cov = X.t() @ X / (X.shape[0] - 1)
    eigenvalues, eigenvectors = t.linalg.eigh(cov)
    sorted_indices = t.argsort(eigenvalues, descending=True)
    return eigenvectors[:, sorted_indices[:k]]

class MMProbe(t.nn.Module):
    def __init__(self, direction, covariance=None, atol=1e-3, bias=None):
        super().__init__()
        self.direction = t.nn.Parameter(direction, requires_grad=False)
        self.bias = t.nn.Parameter(bias, requires_grad=False) if bias is not None else None
        self.inv = t.nn.Parameter(t.linalg.pinv(covariance, hermitian=True, atol=atol), requires_grad=False) if covariance is not None else None
    def forward(self, x, iid=False):
        if self.bias is not None:
            x = x - self.bias
        return t.sigmoid(x @ self.inv @ self.direction) if (iid and self.inv is not None) else t.sigmoid(x @ self.direction)
    @staticmethod
    def from_data(acts, labels, device="cpu", bias=True):
        acts, labels = acts.to(device), labels.to(device)
        pos_acts, neg_acts = acts[labels == 1], acts[labels == 0]
        pos_mean, neg_mean = pos_acts.mean(0), neg_acts.mean(0)
        centered = t.cat([pos_acts - pos_mean, neg_acts - neg_mean], dim=0)
        covariance = centered.t() @ centered / acts.shape[0]
        return MMProbe(pos_mean - neg_mean, covariance=covariance, bias=0.5 * (pos_mean + neg_mean) if bias else None).to(device)

class LRProbe(t.nn.Module):
    def __init__(self, d_in, scaler_mean=None, scaler_scale=None):
        super().__init__()
        self.net = t.nn.Sequential(t.nn.Linear(d_in, 1, bias=False), t.nn.Sigmoid())
        self.register_buffer("scaler_mean", scaler_mean); self.register_buffer("scaler_scale", scaler_scale)
    def forward(self, x):
        if self.scaler_mean is not None:
            x = (x - self.scaler_mean) / self.scaler_scale
        return self.net(x).squeeze(-1)
    @property
    def direction(self):
        return self.net[0].weight.data[0]
    @staticmethod
    def from_data(acts, labels, C=0.1, device="cpu"):
        X, y = acts.cpu().float().numpy(), labels.cpu().float().numpy()
        scaler = StandardScaler(); X_scaled = scaler.fit_transform(X)
        lr_model = LogisticRegression(C=C, random_state=42, fit_intercept=False, max_iter=1000).fit(X_scaled, y)
        probe = LRProbe(acts.shape[-1], t.tensor(scaler.mean_, dtype=t.float32), t.tensor(scaler.scale_, dtype=t.float32)).to(device)
        probe.net[0].weight.data[0] = t.tensor(lr_model.coef_[0], dtype=t.float32).to(device)
        return probe
log = lambda m: print(time.strftime("%H:%M:%S"), m, flush=True)

In [ ]:
# ---------------- model + data ----------------
from datasets import load_dataset
from huggingface_hub import HfApi, hf_hub_download
from transformers import AutoModelForCausalLM, AutoTokenizer
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
api = HfApi(token=HF_TOKEN)
assert HF_TOKEN, "No HF token: Kaggle > Add-ons > Secrets > add HF_TOKEN and tick it for this notebook"
log(f"HF login: {api.whoami()['name']}")      # must be the account that accepted the model's licence

tok = AutoTokenizer.from_pretrained(MODEL, token=HF_TOKEN)
NG = torch.cuda.device_count()     # T4 x2: keep ~5 GB free on GPU 0 (embeddings + output layer live there) and ~1.5 GB on GPU 1 for activations
kw = dict(device_map="auto", max_memory={i: ("9GiB" if i == 0 else "13GiB") for i in range(NG)} if NG > 1 else None, token=HF_TOKEN, attn_implementation="sdpa")
try:
    model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16, **kw)        # T4 has no bf16: fp16
except TypeError:
    model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, **kw)
model.eval().requires_grad_(False)
blocks = model.model.layers
NL, D, NH = len(blocks), model.config.hidden_size, model.config.num_attention_heads
HD = D // NH
log(f"{MODEL}: {NL} layers, d_model {D}, {NH} heads")

ours = {}
for l in open(hf_hub_download(DATA_REPO, "progress/progress.jsonl", repo_type="dataset", token=HF_TOKEN), encoding="utf-8"):
    r = json.loads(l)
    if r.get("annotations") is not None:
        ours[(r["conversation_id"], r["turn"])] = {"conversation": r["conversation"], "annotations": r["annotations"], "source": "ours",
                    "id": f"{r['conversation_id']}-t{r['turn']}", "turn": r["turn"], "group": r["conversation_id"]}
ours = list(ours.values())
groups = sorted({r["group"] for r in ours}); random.Random(SEED).shuffle(groups)
fold_of = {g: i % FOLDS for i, g in enumerate(groups)}
paper = []
for repo in (["obalcells/longfact-annotations", "obalcells/longfact-augmented-annotations"] if N_PAPER_TRAIN else []):
    ds = load_dataset(repo, "Meta-Llama-3.1-8B-Instruct", split="train", token=HF_TOKEN).shuffle(seed=SEED)
    paper += [{"conversation": r["conversation"], "annotations": r["annotations"], "source": "paper", "id": f"{repo}-{i}", "turn": 1, "group": None}
              for i, r in enumerate(ds.select(range(min(N_PAPER_TRAIN, len(ds)))))]
log(f"ours: {len(ours)} answers in {len(groups)} groups, {FOLDS} folds; paper train texts: {len(paper)}")

In [ ]:
# ---------------- token labels: the paper's rules, applied to the LAST answer ----------------
def chat_ids(msgs, gen=False):
    return tok(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=gen), add_special_tokens=False)["input_ids"]

def common_prefix(a, b):
    n = 0
    while n < min(len(a), len(b)) and a[n] == b[n]:
        n += 1
    return n

def label_example(row, max_len=MAX_LEN):
    conv = [{"role": m["role"], "content": m["content"]} for m in row["conversation"]]
    completion, full = conv[-1]["content"], chat_ids(conv)
    start = common_prefix(full, chat_ids(conv[:-1], gen=True))              # first token of the latest answer
    q0 = common_prefix(full, chat_ids(conv[:-2])) if len(conv) > 2 else 1   # first token of the latest user message
    ctx = min(64, max_len // 4)                                             # cut from the left; keep some context and BOS
    lo = max(0, len(full) - max_len)
    if lo > start - ctx:
        lo = max(0, start - ctx)
    ids = full[lo:lo + max_len]
    if lo > 0 and tok.bos_token_id is not None:
        ids = [tok.bos_token_id] + ids[1:]
    comp, ids_t = start - lo, torch.tensor(ids)
    labels = torch.full((len(ids),), -100.0); labels[comp:] = 0.0          # paper: answer tokens default 0, prompt ignored
    weights = torch.ones(len(ids))
    pos, neg, spans, cur = [], [], [], comp
    for a in sorted([a for a in row["annotations"] if a.get("index") is not None], key=lambda a: a["index"]):
        y = _MAP_LABEL_TO_SCALAR.get(a.get("label"), -100.0)
        if y == -100.0 or not a["span"] or a["span"] not in completion:
            continue
        try:                                                                # paper: search after the previous span
            s = find_string_in_tokens(a["span"], ids_t[cur:], tok)
        except (AssertionError, ValueError):
            continue
        a0, a1 = s.start + cur, s.stop + cur - 1
        if a0 < comp:
            continue
        cur = a0
        labels[a0:a1 + 1] = y; weights[a0:a1 + 1] = POS_WEIGHT if y == 1.0 else NEG_WEIGHT
        (pos if y == 1.0 else neg).append([a0, a1]); spans.append((a0, a1, y, a["span"], a.get("label")))
    for a0, a1 in pos:                                                      # positives win overlaps (paper)
        labels[a0:a1 + 1] = 1.0; weights[a0:a1 + 1] = POS_WEIGHT
    seg = {"earlier turns": (1, max(1, q0 - lo)), "current question": (max(1, q0 - lo), comp), "current answer": (comp, len(ids))}
    return {"ids": ids_t, "labels": labels, "weights": weights, "pos": pos, "neg": neg, "spans": spans, "comp": comp, "seg": seg, "row": row}

ex = label_example(ours[0])
print(f"{ex['row']['id']}: {len(ex['ids'])} tokens, answer from token {ex['comp']}, {len(ex['pos'])} hallucinated + {len(ex['neg'])} supported entities labelled")
n_ann = sum(1 for r in ours for a in r["annotations"] if a.get("index") is not None and _MAP_LABEL_TO_SCALAR.get(a.get("label"), -100.0) != -100.0)
n_lab = sum(len(label_example(r)["spans"]) for r in ours)
print(f"all answers: {n_lab} of {n_ann} judged entities found in the tokens and used for training / scoring")

In [ ]:
# ---------------- one forward pass: residual stream, attention output, per-head outputs, next-token entropy ----------------
def get_acts(ids):
    got, hooks = {}, []
    for l in range(NL):
        hooks.append(blocks[l].register_forward_hook(lambda m, a, o, l=l: got.__setitem__(("resid", l), (o[0] if isinstance(o, tuple) else o)[0].detach())))
        hooks.append(blocks[l].self_attn.register_forward_hook(lambda m, a, o, l=l: got.__setitem__(("attn", l), (o[0] if isinstance(o, tuple) else o)[0].detach())))
        hooks.append(blocks[l].self_attn.o_proj.register_forward_pre_hook(lambda m, a, l=l: got.__setitem__(("head", l), a[0][0].detach())))
    try:
        with torch.no_grad():
            h = model.model(input_ids=ids[None].to(model.device)).last_hidden_state[0]   # no full-vocab logits (Aya: 256k x seq = 1+ GB)
            W, scale, ent = model.lm_head, getattr(model, "logit_scale", 1.0), []           # Cohere multiplies logits by logit_scale; Llama has none
            for i in range(0, len(h), 256):                                                  # next-token entropy, 256 positions at a time
                p = (W(h[i:i + 256].to(W.weight.device)).float() * scale).log_softmax(-1)
                ent.append(-(p.exp() * p).sum(-1).cpu())
            ent = torch.cat(ent)
            got["entropy"] = torch.cat([torch.zeros(1), ent[:-1]])                          # entropy of the distribution that produced each token
            del h, p
    finally:
        for h in hooks:
            h.remove()
    return got

# ---------------- trainable probes (the paper's loss), one per layer per family ----------------
class HeadProbe(nn.Module):                            # a separate linear probe on each attention head's output
    def __init__(self):
        super().__init__()
        self.w, self.b = nn.Parameter(torch.randn(NH, HD) * 0.01), nn.Parameter(torch.zeros(NH))
    def forward(self, z):                              # z: [seq, NH * HD] (input of o_proj) -> logits [NH, seq]
        return torch.einsum("shd,hd->hs", z.view(len(z), NH, HD), self.w) + self.b[:, None]

def make_probe(fam, l):
    torch.manual_seed(SEED + l)                        # a different start per layer, so cross-layer cosines are not inflated by a shared init
    if fam in ("lin", "attn"):                         # paper: N(0, 0.01) weights, zero bias
        h = nn.Linear(D, 1); nn.init.normal_(h.weight, 0.0, 0.01); nn.init.zeros_(h.bias); return h
    if fam == "mlp":
        return nn.Sequential(nn.Linear(D, MLP_HIDDEN), nn.GELU(), nn.Linear(MLP_HIDDEN, 1))
    return HeadProbe()
SRC = {"lin": "resid", "mlp": "resid", "attn": "attn", "head": "head"}

def probe_logits(fam, h, x):                           # -> [n, seq] (n = 1, or NH for head probes)
    return h(x.float()) if fam == "head" else h(x.float()).T

def probe_loss(logits, e, omega):
    # paper trainer: (1 - omega) * token BCE + omega * span-max BCE, omega rising 0 -> 1 over training
    n = len(logits)
    lab, w = e["labels"].to(logits.device)[None].expand(n, -1), e["weights"].to(logits.device)[None].expand(n, -1)
    bce = compute_probe_bce_loss(logits, lab, w)
    mx = compute_probe_max_aggregation_loss(logits, lab, w, [e["pos"]] * n, [e["neg"]] * n)
    return (1 - omega) * bce + omega * mx

In [ ]:
# ---------------- train + score, fold by fold (checkpointed) ----------------
def ent_mask(e):
    m = torch.zeros(len(e["ids"]), dtype=torch.bool)
    for a0, a1 in e["pos"] + e["neg"]:
        m[a0:a1 + 1] = True
    return m

LR_CS = (1e-4, 1e-3, 1e-2, 1e-1, 1.0)           # L2 strengths tried for logistic regression (smaller C = stronger)
def pick_C(X, Y, G):
    # entity tokens of one answer are near-duplicates, so the strength is chosen by AUC on held-out ANSWERS (grouped CV inside the training fold)
    cv = GroupKFold(n_splits=min(3, len(set(G))))
    sc = {C: np.nanmean(cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(C=C, fit_intercept=False, max_iter=1000)),
                                        X, Y, groups=G, cv=cv, scoring="roc_auc", error_score=np.nan, n_jobs=-1)) for C in LR_CS}
    sc = {C: v for C, v in sc.items() if not np.isnan(v)}
    return max(sc, key=sc.get) if sc else 0.1     # ARENA's default if no split had both classes

def run_fold(f):
    done_file = f"{OUT}/fold{f}_{RUN}.pt"
    if not os.path.exists(done_file):
        try:
            os.replace(hf_hub_download(OUT_REPO, f"{RUN}/fold{f}.pt", token=HF_TOKEN), done_file)
        except Exception:
            pass
    if os.path.exists(done_file):
        log(f"fold {f}: already done"); return torch.load(done_file, weights_only=False)
    train = [r for r in ours if fold_of[r["group"]] != f] + paper
    test = [r for r in ours if fold_of[r["group"]] == f]
    order = [i for ep in range(EPOCHS) for i in random.Random(SEED + ep).sample(range(len(train)), len(train))]
    total = math.ceil(len(order) / BATCH)
    heads = {(fam, l): make_probe(fam, l).to(next(blocks[l].parameters()).device) for fam in SRC for l in range(NL)}
    params = [p for h in heads.values() for p in h.parameters()]
    opt = torch.optim.AdamW(params, lr=LR, weight_decay=0.0)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: max(0.0, 1 - s / total))    # linear decay, as HF Trainer
    ck, k0 = f"{OUT}/ckpt_{RUN}_fold{f}.pt", 0
    if os.path.exists(ck):
        c = torch.load(ck, map_location="cpu", weights_only=False)
        k0 = c["k"]; opt.load_state_dict(c["opt"]); sched.load_state_dict(c["sched"])
        for key, h in heads.items():
            h.load_state_dict(c["heads"][f"{key[0]}|{key[1]}"])
        log(f"fold {f}: resuming at text {k0}/{len(order)}")
    t0 = time.time()
    for k in range(k0, len(order)):
        row = train[order[k]]
        e = label_example(row)
        if not e["pos"] and not e["neg"]:
            continue
        acts = get_acts(e["ids"])
        omega = min(1.0, (k // BATCH) / total)
        for (fam, l), h in heads.items():
            (probe_loss(probe_logits(fam, h, acts[SRC[fam], l]), e, omega) / BATCH).backward()
        if (k + 1) % BATCH == 0 or k + 1 == len(order):
            for h in heads.values():
                torch.nn.utils.clip_grad_norm_(h.parameters(), 1.0)
            opt.step(); sched.step(); opt.zero_grad()
        del acts
        if (k + 1) % CKPT_EVERY == 0:
            torch.save({"k": k + 1, "opt": opt.state_dict(), "sched": sched.state_dict(),
                        "heads": {f"{a}|{b}": h.state_dict() for (a, b), h in heads.items()}}, ck)
            log(f"fold {f}: {k + 1}/{len(order)} texts, ~{(time.time() - t0) / (k + 1 - k0) * (len(order) - k - 1) / 60:.0f} min left")
    # closed-form / sklearn probes (ARENA) on entity tokens of the training answers: all of ours + CACHE_PAPER paper texts
    cache, cache_y, cache_g, arena, lr_C = {l: [] for l in range(NL)}, [], [], {}, {}
    for gi, row in enumerate([r for r in train if r["source"] == "ours"] + [r for r in train if r["source"] == "paper"][:CACHE_PAPER]):
        e = label_example(row); m = ent_mask(e)
        if m.any():
            acts = get_acts(e["ids"])
            for l in range(NL):
                cache[l].append(acts["resid", l][m.to(acts["resid", l].device)].half().cpu())
            cache_y.append(e["labels"][m]); cache_g.append(torch.full((int(m.sum()),), gi)); del acts
    if cache_y:
        Y, G = torch.cat(cache_y), torch.cat(cache_g).numpy()
        for l in range(NL):
            X = torch.cat(cache[l]).float()
            lr_C[l] = pick_C(X.numpy(), Y.numpy(), G)
            arena["LR", l], arena["MM", l] = LRProbe.from_data(X, Y, C=lr_C[l]), MMProbe.from_data(X, Y)
        log(f"fold {f}: LR / MM fitted on {len(Y)} entity tokens from {len(set(G))} answers at every layer; chosen C per layer: {lr_C}")
    # score the held-out answers: per-token probabilities for every detector, span maxima for every head
    ph = None
    if PAPER_PROBE:
        cfg = json.load(open(hf_hub_download("obalcells/hallucination-probes", f"{PAPER_PROBE}/probe_config.json")))
        ph = nn.Linear(cfg["hidden_size"], 1)
        ph.load_state_dict({k: v.float() for k, v in torch.load(hf_hub_download("obalcells/hallucination-probes", f"{PAPER_PROBE}/probe_head.bin"),
                                                                  map_location="cpu", weights_only=True).items()})
    records = []
    for row in test:
        e = label_example(row)
        if not e["pos"] and not e["neg"]:
            continue
        acts = get_acts(e["ids"])
        with torch.no_grad():
            probs, headmax = {}, torch.zeros(NL, NH, len(e["pos"]) + len(e["neg"]))
            for (fam, l), h in heads.items():
                p = torch.sigmoid(probe_logits(fam, h, acts[SRC[fam], l])).float().cpu()
                if fam == "head":
                    headmax[l] = torch.stack([p[:, a0:a1 + 1].max(1).values for a0, a1 in e["pos"] + e["neg"]], 1)
                else:
                    probs[f"{fam}-L{l}"] = p[0]
            for (name, l), pr in arena.items():
                probs[f"{name}-L{l}"] = pr(acts["resid", l].float().cpu())
            probs["entropy"] = acts["entropy"] / math.log(len(tok))          # scaled to [0, 1]
            if ph is not None:
                x = acts["resid", cfg["layer_idx"]].float().cpu()
                probs[f"paper-L{cfg['layer_idx']}"] = torch.sigmoid(ph(x)).squeeze(-1)
        m = ent_mask(e)
        records.append({"e": e, "probs": probs, "headmax": headmax, "ent_acts": {l: acts["resid", l][m.to(acts["resid", l].device)].half().cpu() for l in range(NL)}})
        del acts
    out = {"records": records, "lr_C": lr_C, "heads": {f"{a}|{b}": h.state_dict() for (a, b), h in heads.items() if a in ("lin", "attn")},
           "arena_dirs": {f"{a}|{b}": (p.direction.float().cpu() / (p.scaler_scale if a == "LR" else 1)) for (a, b), p in arena.items()}}
    torch.save(out, done_file)
    try:
        api.create_repo(OUT_REPO, exist_ok=True)
        api.upload_file(path_or_fileobj=done_file, path_in_repo=f"{RUN}/fold{f}.pt", repo_id=OUT_REPO, commit_message=f"{RUN}: fold {f}")
    except Exception as ex:
        log(f"upload failed (kept locally): {str(ex)[:120]}")
    del heads, opt; gc.collect(); torch.cuda.empty_cache()
    return out

folds = [run_fold(f) for f in range(FOLDS)]
records = [r for fo in folds for r in fo["records"]]
log(f"scored {len(records)} held-out answers")

In [ ]:
# ---------------- metrics (the paper's three levels) for every detector ----------------
def metrics(recs, key):
    out, lv = {"n_answers": len(recs)}, {"all": ([], []), "span": ([], []), "span_max": ([], [])}
    for r in recs:
        e, p = r["e"], r["probs"][key]
        m = e["labels"] != -100.0
        lv["all"][0].append(p[m]); lv["all"][1].append(e["labels"][m])
        for y, sp in ((1.0, e["pos"]), (0.0, e["neg"])):
            for a0, a1 in sp:
                lv["span"][0].append(p[a0:a1 + 1]); lv["span"][1].append(torch.full((a1 - a0 + 1,), y))
                lv["span_max"][0].append(p[a0:a1 + 1].max()[None]); lv["span_max"][1].append(torch.tensor([y]))
    for name, (ps, ys) in lv.items():
        pr, yy = torch.cat(ps).numpy(), torch.cat(ys).numpy()
        for mk, mv in compute_clf_metrics((pr > 0.5).astype(float), yy, pr).items():
            out[f"{name}_{mk}"] = mv
    return out

sets = {"all": records}
for tn in (1, 2, 3):
    sets[f"turn {tn}"] = [r for r in records if r["e"]["row"]["turn"] == tn]
keys = list(records[0]["probs"])
res = pd.DataFrame([{"detector": k, "family": k.rsplit("-L", 1)[0], "layer": int(k.rsplit("-L", 1)[1]) if "-L" in k else -1, "set": s, **metrics(rs, k)}
                    for k in keys for s, rs in sets.items() if rs])
# per-head span-max AUC (out of fold)
ys = torch.cat([torch.tensor([1.0] * len(r["e"]["pos"]) + [0.0] * len(r["e"]["neg"])) for r in records]).numpy()
hm = torch.cat([r["headmax"] for r in records], 2)
head_auc = np.array([[compute_clf_metrics((hm[l, h].numpy() > 0.5).astype(float), ys, hm[l, h].numpy())["auc"] for h in range(NH)] for l in range(NL)])
res.to_csv(f"{OUT}/results_{RUN}.csv", index=False); np.save(f"{OUT}/head_auc_{RUN}.npy", head_auc)
A = res[res.set == "all"]
BEST_LAYER = int(A[A.family == "lin"].sort_values("span_max_auc").iloc[-1].layer)
best = A.sort_values("span_max_auc", ascending=False).groupby("family").head(1)
COLS = ["span_max_auc", "span_max_recall_at_0.1_fpr", "span_max_f1", "span_auc", "all_auc", "all_recall_at_0.1_fpr", "all_f1"]
print(f"best layer for the linear probe: {BEST_LAYER}")
display(best[["detector"] + COLS].round(3))

In [ ]:
# ---------------- figures (300 dpi, saved to figs/) ----------------
def savefig(name):
    plt.tight_layout(); plt.savefig(f"{OUT}/figs/{RUN}_{name}.png", dpi=300); plt.show()
COL = {"lin": "#d33a2c", "mlp": "#9b72b8", "attn": "#c9a227", "LR": "#2e9b45", "MM": "#4f8a63", "entropy": "#8a8f96", "paper": "#111111", "head": "#6f8fb8"}

# 1. which layer carries the most hallucination information? (span-max AUC per layer, every detector; head = best head of the layer)
fig, ax = plt.subplots(figsize=(8, 4))
for fam in ("lin", "mlp", "attn", "LR", "MM"):
    d = A[A.family == fam].sort_values("layer")
    ax.plot(d.layer, d.span_max_auc, "-o", ms=3, color=COL[fam], label=fam)
ax.plot(range(NL), head_auc.max(1), "--", color=COL["head"], label="best single head")
ax.axhline(A[A.family == "entropy"].span_max_auc.iloc[0], color=COL["entropy"], ls="-.", label="entropy (no training)")
for _, r in A[A.family == "paper"].iterrows():
    ax.plot(r.layer, r.span_max_auc, "*", ms=13, color=COL["paper"], label="paper's released probe")
ax.axhline(0.5, ls=":", color="k", lw=1); ax.set_xlabel("layer"); ax.set_ylabel("span-max AUC (held-out)"); ax.set_ylim(0.3, 1)
ax.legend(fontsize=7, ncol=3); ax.set_title(f"{RUN}: hallucination information by layer"); savefig("layers")

# 2. every paper metric, best layer of each detector
fig, ax = plt.subplots(figsize=(9, 3.8)); w = 0.8 / len(best)
for j, (_, r) in enumerate(best.iterrows()):
    ax.bar(np.arange(len(COLS)) + j * w, [r[c] for c in COLS], w, color=COL[r.family], label=r.detector)
ax.set_xticks(np.arange(len(COLS)) + 0.4 - w / 2); ax.set_xticklabels([c.replace("_", " ") for c in COLS], fontsize=7, rotation=12)
ax.set_ylim(0, 1); ax.legend(fontsize=7, ncol=4); ax.set_title(f"{RUN}: paper metrics, best layer of each detector"); savefig("metrics")

# 3. by turn: best layer of each family
d = res[res.set.str.startswith("turn") & res.detector.isin(best.detector)]
d.pivot_table(index="set", columns="detector", values="span_max_auc").plot.bar(figsize=(7.5, 3.6), rot=0, color=[COL[k.rsplit("-L", 1)[0]] for k in sorted(d.detector.unique())])
plt.axhline(0.5, ls=":", color="k", lw=1); plt.ylim(0, 1); plt.ylabel("span-max AUC"); plt.title(f"{RUN}: by conversation turn"); plt.legend(fontsize=7); savefig("by_turn")

# 4. which attention heads carry it? per-head probe AUC (left) and each head's direct push along the linear probe direction (right)
fig, axs = plt.subplots(1, 2, figsize=(11, 4.4))
im = axs[0].imshow(head_auc, aspect="auto", cmap="viridis", vmin=0.5, vmax=max(0.51, np.nanmax(head_auc)), origin="lower")
axs[0].set_xlabel("head"); axs[0].set_ylabel("layer"); axs[0].set_title("span-max AUC of a probe on each head's output"); plt.colorbar(im, ax=axs[0])
#    direct contribution: head output -> o_proj slice -> dot with the lin probe direction at BEST_LAYER (fold 0), hallucinated minus supported
wdir = folds[0]["heads"][f"lin|{BEST_LAYER}"]["weight"][0].float()
contrib, n = {1: torch.zeros(BEST_LAYER + 1, NH), 0: torch.zeros(BEST_LAYER + 1, NH)}, {1: 0, 0: 0}
for r in [r for r in folds[0]["records"]][:ATTN_EXAMPLES]:
    e = label_example(r["e"]["row"], max_len=ATTN_WINDOW)
    acts = get_acts(e["ids"])
    for l in range(BEST_LAYER + 1):
        Wo = blocks[l].self_attn.o_proj.weight.float()                       # [D, NH * HD]
        z = acts["head", l].float().view(-1, NH, HD)                          # [seq, NH, HD]
        per_head = torch.einsum("shd,ehd->she", z, Wo.view(D, NH, HD).to(z.device)) @ wdir.to(z.device)   # [seq, NH]
        for y, sp in ((1, e["pos"]), (0, e["neg"])):
            for a0, a1 in sp:
                contrib[y][l] += per_head[a0:a1 + 1].mean(0).cpu(); n[y] += 1 if l == 0 else 0
    del acts
diff = contrib[1] / max(1, n[1]) - contrib[0] / max(1, n[0])
v = float(diff.abs().max())
im = axs[1].imshow(diff.numpy(), aspect="auto", cmap="RdBu_r", vmin=-v, vmax=v, origin="lower")
axs[1].set_xlabel("head"); axs[1].set_ylabel("layer"); plt.colorbar(im, ax=axs[1])
axs[1].set_title(f"head → layer-{BEST_LAYER} probe direction\n(hallucinated − supported)"); savefig("heads")
TOP_HEADS = [(int(i // NH), int(i % NH)) for i in torch.argsort(diff.flatten(), descending=True)[:5]]
print("heads that push hardest toward 'hallucinated' (layer, head):", TOP_HEADS)

# 5. PCA of entity tokens at BEST_LAYER (held-out answers) + projection on the probe direction
X = torch.cat([r["ent_acts"][BEST_LAYER] for r in records]).float()
Y = torch.cat([r["e"]["labels"][torch.cat([torch.arange(a0, a1 + 1) for a0, a1 in r["e"]["pos"] + r["e"]["neg"]]).unique()] for r in records])
pcs = get_pca_components(X, 2); Z = (X - X.mean(0)) @ pcs
fig, axs = plt.subplots(1, 2, figsize=(10, 3.9))
for y, c, nme in ((0, "#2e9b45", "Supported"), (1, "#d33a2c", "Not Supported / Insufficient")):
    axs[0].scatter(Z[Y == y, 0], Z[Y == y, 1], s=7, alpha=0.5, color=c, label=nme)
    axs[1].hist((X @ wdir)[Y == y].numpy(), bins=40, alpha=0.6, color=c, density=True, label=nme)
axs[0].set_title(f"PCA of entity tokens, layer {BEST_LAYER}"); axs[0].legend(fontsize=7)
axs[1].set_title(f"projection on the linear probe direction, layer {BEST_LAYER}"); savefig("pca")

# 6. do the detectors find the same direction? (fold 0) — lin across layers, and lin vs attn / LR / MM at BEST_LAYER
dirs = torch.stack([folds[0]["heads"][f"lin|{l}"]["weight"][0].float() for l in range(NL)]); dirs = dirs / dirs.norm(dim=1, keepdim=True)
fig, axs = plt.subplots(1, 2, figsize=(10.5, 4.3))
im = axs[0].imshow((dirs @ dirs.T).numpy(), cmap="RdBu_r", vmin=-1, vmax=1, origin="lower"); plt.colorbar(im, ax=axs[0])
axs[0].set_xlabel("layer"); axs[0].set_ylabel("layer"); axs[0].set_title("cosine of linear-probe directions across layers")
cand = {"lin": folds[0]["heads"][f"lin|{BEST_LAYER}"]["weight"][0].float(), "attn": folds[0]["heads"][f"attn|{BEST_LAYER}"]["weight"][0].float()}
for a in ("LR", "MM"):
    if f"{a}|{BEST_LAYER}" in folds[0]["arena_dirs"]:
        cand[a] = folds[0]["arena_dirs"][f"{a}|{BEST_LAYER}"]
if PAPER_PROBE and BEST_LAYER == 30:
    cand["paper"] = torch.load(hf_hub_download("obalcells/hallucination-probes", f"{PAPER_PROBE}/probe_head.bin"), weights_only=True)["weight"][0].float()
Wc = torch.stack(list(cand.values())); Wc = Wc / Wc.norm(dim=1, keepdim=True)
im = axs[1].imshow((Wc @ Wc.T).numpy(), cmap="RdBu_r", vmin=-1, vmax=1); plt.colorbar(im, ax=axs[1])
axs[1].set_xticks(range(len(cand))); axs[1].set_xticklabels(cand); axs[1].set_yticks(range(len(cand))); axs[1].set_yticklabels(cand)
axs[1].set_title(f"cosine between detectors, layer {BEST_LAYER}"); savefig("directions")

In [ ]:
# ---------------- stage 4 view: what the judge tagged vs what the probe lights up (per word) ----------------
# background = linear-probe score at BEST_LAYER (red above the probe's best span-max threshold, THR), underline = web-search judge label (red = Not Supported / Insufficient, green = Supported).
# Scores are shown per word because one word (and one Devanagari character) can span several tokens.
def word_view(e, probs):
    ids, comp = e["ids"].tolist(), e["comp"]
    ends = [len(tok.decode(ids[comp:j + 1], skip_special_tokens=True)) for j in range(comp, len(ids))]
    text = tok.decode(ids[comp:], skip_special_tokens=True)
    lab = {j - comp: y for a0, a1, y, _, _ in e["spans"] for j in range(a0, a1 + 1)}
    out = []
    for m in re.finditer(r"\S+|\s+", text):
        s0, s1 = m.span()
        if m.group().isspace():
            out.append(htmllib.escape(m.group()).replace("\n", "<br/>")); continue
        tj = [j for j in range(len(ends)) if (ends[j - 1] if j else 0) < s1 and ends[j] > s0]
        sc, ls = max([float(probs[comp + j]) for j in tj], default=0.0), {lab[j] for j in tj if j in lab}
        bg = f"rgba(211,58,44,{0.25 + 0.75 * (sc - THR) / max(1e-6, 1 - THR):.2f})" if sc >= THR else "transparent"
        ul = f"text-decoration:underline;text-decoration-thickness:2.5px;text-decoration-color:{'#d33a2c' if 1.0 in ls else '#2e9b45'};" if ls else ""
        out.append(f'<span title="{sc:.2f}" style="background:{bg};{ul}">{htmllib.escape(m.group())}</span>')
    return "<div style=\"font-family:'Noto Sans Devanagari','Mangal',sans-serif;font-size:15px;line-height:1.9;max-width:900px\">" + "".join(out) + "</div>"

SHOW = f"lin-L{BEST_LAYER}"
THR = float(A[A.detector == SHOW].span_max_optimal_threshold.iloc[0])
print(f"{SHOW}: words above {THR:.3f} (the span-max threshold that maximises accuracy) are shaded")
conv = next(r["e"]["row"]["group"] for r in records if r["e"]["row"]["turn"] == 3)
SHOW_RECS = sorted([r for r in records if r["e"]["row"]["group"] == conv], key=lambda r: r["e"]["row"]["turn"])
for r in SHOW_RECS:
    e = r["e"]
    page = f"<meta charset='utf-8'><h4>{e['row']['id']} · {SHOW}</h4>" + word_view(e, r["probs"][SHOW])
    open(f"{OUT}/figs/{RUN}_view_{e['row']['id']}.html", "w", encoding="utf-8").write(page); display(HTML(page))
    display(pd.DataFrame([{"entity": s[3], "judge": s[4], "probe max": round(float(r["probs"][SHOW][s[0]:s[1] + 1].max()), 3),
                           "agree": bool(r["probs"][SHOW][s[0]:s[1] + 1].max() >= THR) == (s[2] == 1.0)} for s in e["spans"]]))

In [ ]:
# ---------------- attention: where hallucinated vs supported entity tokens look, by layer + the top head's pattern ----------------
import circuitsvis as cv
try:
    model.set_attn_implementation("eager")             # attention weights need eager attention
except Exception:
    model.config._attn_implementation = "eager"
SEGS = ("earlier turns", "current question", "current answer")
mass, cnt = {(y, s): np.zeros(NL) for y in (0, 1) for s in SEGS}, {0: 0, 1: 0}
recs = [r for r in records if r["e"]["row"]["turn"] >= (2 if SETTING == "multiturn" else 1)][:ATTN_EXAMPLES]
for r in recs:
    e = label_example(r["e"]["row"], max_len=ATTN_WINDOW)
    with torch.no_grad():
        att = model.model(input_ids=e["ids"][None].to(model.device), output_attentions=True).attentions   # per layer [1, NH, q, k]
    for y, sp in ((1, e["pos"]), (0, e["neg"])):
        q = [i for a0, a1 in sp for i in range(a0, a1 + 1)]
        if q:
            cnt[y] += 1
            for l, Aw in enumerate(att):
                a = Aw[0].float().mean(0)[q].cpu()
                for s in SEGS:
                    s0, s1 = e["seg"][s]; mass[y, s][l] += a[:, s0:s1].sum(1).mean().item()
    del att; torch.cuda.empty_cache()
fig, ax = plt.subplots(figsize=(7, 3.6))
for (y, s), v in mass.items():
    if cnt[y]:
        ax.plot(v / cnt[y], "-" if y else "--", color={"earlier turns": "#6f8fb8", "current question": "#c9a227", "current answer": "#8a8f96"}[s],
                label=f"{'hallucinated' if y else 'supported'} → {s}")
ax.set_xlabel("layer"); ax.set_ylabel("attention mass (head-averaged)"); ax.legend(fontsize=7); ax.set_title(f"{RUN}: where entity tokens attend"); savefig("attention")

# the attention pattern of the head that pushes hardest toward 'hallucinated', just before the first hallucinated entity
L0, H0 = TOP_HEADS[0]
r = next(r for r in recs if r["e"]["pos"])
e = label_example(r["e"]["row"], max_len=ATTN_WINDOW); a0 = e["pos"][0][0]; win = slice(max(0, a0 - 60), a0 + 10)
with torch.no_grad():
    Aw = model.model(input_ids=e["ids"][None].to(model.device), output_attentions=True).attentions[L0][0][:, win, win].float().cpu()
page = cv.attention.attention_heads(tokens=[tok.decode([i]) for i in e["ids"][win]], attention=Aw, attention_head_names=[f"L{L0}H{h}" for h in range(NH)])
open(f"{OUT}/figs/{RUN}_attention_L{L0}.html", "w", encoding="utf-8").write(str(page)); display(page)
print(f"look at head L{L0}H{H0} in the viewer")
try:
    model.set_attn_implementation("sdpa")
except Exception:
    model.config._attn_implementation = "sdpa"

In [ ]:
# ---------------- upload results + figures ----------------
try:
    api.upload_folder(folder_path=OUT, repo_id=OUT_REPO, path_in_repo=RUN, allow_patterns=["*.csv", "*.npy", "figs/*"], commit_message=f"{RUN}: results, figures")
    log(f"uploaded to https://huggingface.co/{OUT_REPO}/tree/main/{RUN}")
except Exception as ex:
    log(f"upload failed (files are in {OUT}): {ex}")